# Phase 1 — Conformité : vérifier le droit de crawler
**Durée estimée : 30 min**

---

## Objectif
Avant d'écrire une seule ligne de code de scraping, un chef de projet IA doit **prouver que la collecte est autorisée**.  
Dans ce notebook, vous allez analyser les fichiers `robots.txt` de 3 sites et produire un tableau de décision argumenté.

## Ce que vous rendez à la fin
- Un tableau comparatif (3 lignes × 4 colonnes)
- Une décision argumentée : quel site scraper ? pourquoi ?

---
> **Rappel juridique rapide**  
> Un fichier `robots.txt` n'a **pas de valeur légale contraignante** (c'est une convention technique), mais l'ignorer délibérément peut être retenu contre vous dans une procédure (CJUE, Ryanair/PR Aviation, 2015). Les CGU, elles, ont une valeur contractuelle.

In [ ]:
# --- Installation (décommentez si vous êtes sur Google Colab) ---
# !pip install requests pandas

In [ ]:
import requests
import pandas as pd
from urllib.robotparser import RobotFileParser
from IPython.display import display, Markdown

print("Imports OK ✓")

---
## Étape 1 — Récupérer et afficher les robots.txt

Exécutez la cellule ci-dessous pour télécharger et afficher le contenu brut des 3 fichiers.  
**Lisez-les attentivement** avant de répondre aux questions.

In [ ]:
SITES = {
    "data.gouv.fr": "https://www.data.gouv.fr/robots.txt",
    "lemonde.fr":   "https://www.lemonde.fr/robots.txt",
    "wikipedia.fr": "https://fr.wikipedia.org/robots.txt",
}

robots_raw = {}

for nom, url in SITES.items():
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        robots_raw[nom] = r.text
        print(f"✅ {nom} — {len(r.text)} caractères récupérés")
    except Exception as e:
        robots_raw[nom] = ""
        print(f"❌ {nom} — Erreur : {e}")

In [ ]:
# Affiche le robots.txt d'un site — changez le nom pour voir les autres
SITE_A_AFFICHER = "data.gouv.fr"  # <-- modifiez ici : "lemonde.fr" ou "wikipedia.fr"

print(f"=== robots.txt de {SITE_A_AFFICHER} ===")
print(robots_raw[SITE_A_AFFICHER][:3000])  # on limite à 3000 chars pour la lisibilité
print("[...]") if len(robots_raw[SITE_A_AFFICHER]) > 3000 else None

---
## Étape 2 — Tester des URLs spécifiques

La bibliothèque `urllib.robotparser` intégrée à Python peut lire un robots.txt et répondre automatiquement : "cette URL est-elle autorisée pour User-agent: *  ?"

Nous allons tester 2 URLs par site :
- une **URL "safe"** (page de contenu standard)
- une **URL "risquée"** (page de recherche, paramètre `?q=`, etc.)

In [ ]:
TESTS = {
    "data.gouv.fr": (
        "https://www.data.gouv.fr/robots.txt",
        [
            "https://www.data.gouv.fr/fr/datasets/",               # page de liste — safe
            "https://www.data.gouv.fr/fr/search/?q=education",     # recherche — risquée
        ]
    ),
    "lemonde.fr": (
        "https://www.lemonde.fr/robots.txt",
        [
            "https://www.lemonde.fr/economie/",                    # rubrique — safe
            "https://www.lemonde.fr/recherche/?keywords=ia",       # recherche — risquée
        ]
    ),
    "wikipedia.fr": (
        "https://fr.wikipedia.org/robots.txt",
        [
            "https://fr.wikipedia.org/wiki/Intelligence_artificielle",  # article — safe
            "https://fr.wikipedia.org/w/index.php?search=IA",          # recherche interne — risquée
        ]
    ),
}

def lire_robots(robots_url: str) -> RobotFileParser:
    """Charge un robots.txt en gérant le BOM UTF-8 (ex. Wikipedia).
    
    Note : urllib.robotparser.read() ne strip pas le BOM ﻿,
    ce qui fait échouer silencieusement le parsing → can_fetch() retourne False partout.
    On parse manuellement après avoir récupéré et nettoyé le contenu.
    """
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        r = requests.get(robots_url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        contenu_propre = r.text.lstrip("﻿")  # strip BOM si présent
        rp.parse(contenu_propre.splitlines())
    except Exception as e:
        print(f"  ⚠️  Impossible de lire {robots_url} : {e}")
    return rp

resultats = []

for site, (robots_url, urls) in TESTS.items():
    rp = lire_robots(robots_url)
    for url in urls:
        autorisee = rp.can_fetch("*", url)
        delay     = rp.crawl_delay("*")
        resultats.append({
            "Site":        site,
            "URL testée":  url,
            "Autorisée ?": "✅ Oui" if autorisee else "❌ Non",
            "Crawl-delay": str(delay) + "s" if delay else "Non défini",
        })

df_tests = pd.DataFrame(resultats)
display(df_tests)

# Affiche les BOM détectés pour pédagogie
print("\n📌 BOM UTF-8 détecté sur :")
for nom, url in [("data.gouv.fr", "https://www.data.gouv.fr/robots.txt"),
                 ("lemonde.fr",   "https://www.lemonde.fr/robots.txt"),
                 ("wikipedia.fr", "https://fr.wikipedia.org/robots.txt")]:
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        bom = r.text.startswith("﻿")
        print(f"  {nom} : {'⚠️  OUI → parsing manuel nécessaire' if bom else 'Non'}")
    except:
        pass

---
## Comment lire ce tableau ?

### La colonne "Autorisée ?"

`can_fetch("*", url)` simule la question : *"un robot anonyme a-t-il le droit de visiter cette URL ?"*  
La réponse vient des règles `Allow` / `Disallow` du bloc `User-agent: *` du robots.txt.

**Points de vigilance sur nos résultats :**

| Ligne | Résultat | Ce qu'il faut retenir |
|-------|----------|-----------------------|
| data.gouv.fr `/fr/search/?q=` | ✅ Oui | Le robots.txt interdit `/fr/datasets/search?*` (avec le chemin complet `datasets`), mais **pas** `/fr/search/`. Subtilité à signaler dans votre analyse. |
| lemonde.fr `/recherche/` | ✅ Oui | Le robots.txt du Monde n'interdit pas la recherche pour `User-agent: *`. **Mais robots.txt autorisé ≠ scraping faisable** : Le Monde utilise JavaScript dynamique et des tokens anti-bot. L'autorisation technique ne garantit pas la faisabilité. |
| wikipedia `/wiki/…` | ✅ Oui | Autorisé — mais **sans le fix BOM ci-dessous, ce résultat affichait ❌ à tort**. |
| wikipedia `/w/index.php?search=` | ❌ Non | Correct : Wikipedia interdit explicitement `/w/` pour `User-agent: *` (`Disallow: /w/`). |

**La règle d'or** : robots.txt vous dit ce que le site *tolère techniquement*, les CGU vous disent ce que vous avez *le droit* de faire, et les protections anti-bot vous disent ce qui est *réellement faisable*.

---

### Qu'est-ce que le BOM UTF-8 ?

**BOM** = *Byte Order Mark* = caractère invisible `U+FEFF` (`﻿`) placé au tout début de certains fichiers texte.  
Il sert à indiquer l'encodage du fichier (UTF-8 ici), mais est souvent inutile — et parfois nuisible.

```
Fichier normal  :  U ser-agent: * ...
Fichier avec BOM:  ﻿ U ser-agent: * ...
                   ↑ caractère invisible U+FEFF
```

**Pourquoi ça casse `urllib.robotparser` ?**

La bibliothèque standard Python lit le robots.txt **ligne par ligne** et cherche exactement les tokens `User-agent`, `Disallow`, `Allow`…  
Si la première ligne commence par `﻿#` au lieu de `#`, le parser ne reconnaît plus le format et **abandonne silencieusement** (sans erreur, sans avertissement) → `can_fetch()` retourne `False` pour toutes les URLs.

```python
# Comportement sans fix (résultat trompeur) :
rp = RobotFileParser()
rp.set_url("https://fr.wikipedia.org/robots.txt")
rp.read()  # ← lit le BOM, échoue silencieusement
rp.can_fetch("*", "https://fr.wikipedia.org/wiki/Python")
# → False  ← FAUX ! L'article est autorisé.

# Comportement avec fix :
contenu = requests.get(...).text.lstrip("﻿")  # strip du BOM
rp.parse(contenu.splitlines())
rp.can_fetch("*", "https://fr.wikipedia.org/wiki/Python")
# → True   ← correct
```

**Leçon chef de projet** : un outil peut donner un résultat "propre" (pas d'erreur) tout en étant silencieusement faux. C'est pourquoi on vérifie toujours manuellement un échantillon de résultats — ici, vérifier sur le robots.txt brut que `/wiki/` n'est pas interdit.

---
## Étape 3 — Tableau comparatif (à compléter)

Complétez le dictionnaire `votre_analyse` ci-dessous avec vos conclusions.  
**Soyez précis** : citez des règles réelles trouvées dans les robots.txt.

In [ ]:
# ✏️  COMPLÉTEZ CE DICTIONNAIRE avec vos observations
votre_analyse = [
    {
        "Site":             "data.gouv.fr",
        "Pages interdites": "À compléter — ex: /fr/search/ est-elle bloquée ?",
        "Crawl-delay":      "À compléter — valeur trouvée ou pause que vous appliquerez",
        "Risque estimé":    "À compléter — Faible / Moyen / Élevé + 1 argument",
    },
    {
        "Site":             "lemonde.fr",
        "Pages interdites": "À compléter",
        "Crawl-delay":      "À compléter",
        "Risque estimé":    "À compléter",
    },
    {
        "Site":             "fr.wikipedia.org",
        "Pages interdites": "À compléter",
        "Crawl-delay":      "À compléter",
        "Risque estimé":    "À compléter",
    },
]

df_analyse = pd.DataFrame(votre_analyse)
display(df_analyse)

---
## Étape 4 — Décision Chef de Projet

Répondez aux 3 questions ci-dessous dans la cellule Markdown suivante.

### Votre décision (à rédiger ici)

**1. Classement du moins risqué au plus risqué :**

| Rang | Site | Argument principal |
|------|------|--------------------|
| 1 (le moins risqué) | *à compléter* | *à compléter* |
| 2 | *à compléter* | *à compléter* |
| 3 (le plus risqué) | *à compléter* | *à compléter* |

**2. Site retenu pour la phase Collecte :**  
*à compléter — ex: Wikipedia, parce que...*

**3. Pause que vous appliquerez entre les requêtes :**  
*à compléter — ex: 2 secondes car le robots.txt ne précise pas de crawl-delay*

---
## ✅ Fin de la Phase 1

Avant de passer au notebook `02_collecte.ipynb`, vérifiez :
- [ ] Votre tableau comparatif est rempli avec des arguments tirés du robots.txt réel
- [ ] Vous avez testé au moins 2 URLs par site
- [ ] Vous avez choisi un site et justifié votre choix

**Passez maintenant à `02_collecte.ipynb` →**